In [6]:
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI
import os

client = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY")
)

In [9]:
response = client.chat.completions.create(
    model="anthropic.claude-3.7-sonnet",
    messages=[
        {"role": "system", "content": "you only respond in Pirate"},
        {"role": "user", "content": "Hi, I'm Zach"},
    ],
    temperature=0.7,
    top_p=1,
)

print(response.choices[0].message.content)

Ahoy there, Zach me hearty!

Cap'n Blackbeard at yer service! *adjusts tricorn hat*

'Tis a fine day to be sailin' the digital seas, ain't it? What brings ye to converse with an old sea dog like meself? Be ye searchin' for buried treasure or just lookin' to share a tale over a mug o' grog?

Speak up now, lad! These salt-crusted ears be waitin' to hear what ye have to say!


In [8]:
class LLMNotesPromptGenerator:
    """Use an LLM to analyze transcript and generate a custom prompt for note creation"""
    
    def analyze_transcript_with_llm(self, transcript: str) -> str:
        """Use LLM to analyze the transcript and identify key elements"""
        analysis_prompt = f"""
        Analyze this lecture transcript and identify the following elements:

        1. MAIN TOPICS: What are the main topics or themes discussed?
        2. TECHNICAL CONCEPTS: What technical terms, APIs, or tools are mentioned?
        3. CODE ELEMENTS: Are there any code snippets, programming concepts, or technical procedures?
        4. LEARNING OBJECTIVES: What should students learn from this lecture?
        5. STRUCTURE: How is the content organized? (e.g., intro, examples, Q&A, announcements)
        6. KEY TAKEAWAYS: What are the most important points students should remember?
        7. ASSIGNMENTS/LOGISTICS: Any homework, deadlines, or course logistics mentioned?

        Be specific and detailed in your analysis. This will be used to create a custom prompt for generating structured lecture notes.

        TRANSCRIPT:
        {transcript}
        """
        
        try:
            response = client.chat.completions.create(
                model="anthropic.claude-3.7-sonnet",
                messages=[
                    {"role": "system", "content": "You are an expert at analyzing educational content and identifying key learning elements."},
                    {"role": "user", "content": analysis_prompt}
                ],
                temperature=0.7,
                top_p=1
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error analyzing transcript: {str(e)}"
    
    def generate_custom_notes_prompt(self, transcript: str) -> str:
        """Generate a custom prompt for creating structured notes based on LLM analysis"""
        
        # First, analyze the transcript with the LLM
        analysis = self.analyze_transcript_with_llm(transcript)
        
        # Then create a custom prompt based on the analysis
        prompt_creation_request = f"""
        Based on this analysis of a lecture transcript, create a detailed prompt that will instruct an LLM to generate well-structured, comprehensive lecture notes.

        TRANSCRIPT ANALYSIS:
        {analysis}

        Create a prompt that:
        1. Incorporates the specific topics and concepts identified in the analysis
        2. Requests appropriate formatting for the type of content detected
        3. Ensures all important elements are captured in the notes
        4. Provides clear structure and organization instructions
        5. Includes specific requirements for technical content, code, Q&A, etc.

        The prompt should be comprehensive and tailored to this specific lecture content. Start the prompt with "Please create comprehensive lecture notes from the following transcript..." and include specific formatting and content requirements based on what you identified in the analysis.
        """
        
        try:
            response = client.chat.completions.create(
                model="anthropic.claude-3.7-sonnet",
                messages=[
                    {"role": "system", "content": "You are an expert at creating educational prompts that generate high-quality structured content."},
                    {"role": "user", "content": prompt_creation_request}
                ],
                temperature=0.7,
                top_p=1
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error generating custom prompt: {str(e)}"

    def process_transcript_file(self, file_path: str) -> dict:
        """Complete workflow: read file, analyze, and generate custom prompt"""
        try:
            # Read the transcript file
            with open(file_path, 'r', encoding='utf-8') as f:
                transcript = f.read()
            
            print(f"Transcript loaded: {len(transcript)} characters")
            print("Analyzing transcript with LLM...")
            
            # Get LLM analysis
            analysis = self.analyze_transcript_with_llm(transcript)
            
            print("Generating custom notes prompt...")
            
            # Generate custom prompt
            custom_prompt = self.generate_custom_notes_prompt(transcript)
            
            return {
                'transcript_length': len(transcript),
                'analysis': analysis,
                'custom_prompt': custom_prompt,
                'full_transcript': transcript
            }
            
        except Exception as e:
            return {'error': f"Error processing file: {str(e)}"}

def generate_notes_prompt_for_transcript(file_path: str = 'data/combined_transcript.txt'):
    """Main function to generate a custom prompt for your transcript"""
    generator = LLMNotesPromptGenerator()
    result = generator.process_transcript_file(file_path)
    
    if 'error' in result:
        print(result['error'])
        return None
    
    print("=" * 60)
    print("TRANSCRIPT ANALYSIS:")
    print("=" * 60)
    print(result['analysis'])
    print("\n" + "=" * 60)
    print("CUSTOM NOTES PROMPT:")
    print("=" * 60)
    print(result['custom_prompt'])
    
    return result

result = generate_notes_prompt_for_transcript()

Transcript loaded: 42558 characters
Analyzing transcript with LLM...
Generating custom notes prompt...
TRANSCRIPT ANALYSIS:
# Analysis of Lecture Transcript

## 1. MAIN TOPICS
- Introduction to AI image generation (NanoBanana/Gemini)
- OpenAI SDK and API usage for AI applications
- Differences between using AI via API vs. ChatGPT interface
- Conversation memory management in AI applications
- System prompts vs. developer prompts
- AI pricing models and token economics

## 2. TECHNICAL CONCEPTS
- OpenAI SDK and Responses API
- Chat completions API
- System prompts vs. developer prompts
- Token pricing models (input vs. output tokens)
- Visual Studio Code with dev containers
- Jupyter notebooks
- Proxy platforms for accessing multiple AI models (OpenAI, DeepSeek, Entropic, Google)
- Temperature parameter for controlling AI creativity
- Message roles (system, user, assistant, developer)
- Conversation memory and context management
- BlankChain/LangChain frameworks

## 3. CODE ELEMENTS
- P